# MoodNote-AI — Fine-tune PhoBERT trên Google Colab

Pipeline phân loại cảm xúc tiếng Việt với **PhoBERT + UIT-VSMEC**.

**Trước khi chạy:**
1. Vào `Runtime → Change runtime type → T4 GPU`
2. Thay `REPO_URL` ở Cell 3 bằng URL GitHub của bạn
3. Chạy từng cell theo thứ tự

## Cell 1 — Kiểm tra GPU

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Không tìm thấy GPU!\n"
        "Vào Runtime → Change runtime type → chọn T4 GPU rồi chạy lại."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU   : {gpu_name}")
print(f"VRAM  : {gpu_mem:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print("Sẵn sàng train!")

GPU   : Tesla T4
VRAM  : 14.6 GB
PyTorch: 2.10.0+cu128
Sẵn sàng train!


## Cell 2 — Mount Google Drive

Model và checkpoint sẽ được lưu vào Drive để không mất khi session kết thúc.

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Tạo thư mục lưu trữ trong Drive
DRIVE_BASE = '/content/drive/MyDrive/MoodNote-AI'
os.makedirs(f'{DRIVE_BASE}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/best_model',  exist_ok=True)

print(f"Drive mounted. Thư mục lưu: {DRIVE_BASE}")

Mounted at /content/drive
Drive mounted. Thư mục lưu: /content/drive/MyDrive/MoodNote-AI


## Cell 3 — Clone repo & cài dependencies

> **Thay `REPO_URL`** bằng URL GitHub của bạn, ví dụ:
> `https://github.com/username/MoodNote-AI.git`

In [2]:
import sys

REPO_URL  = 'https://github.com/ToanHuynh0201/MoodNote-AI.git'  # <-- ĐỔI Ở ĐÂY
REPO_DIR  = '/content/MoodNote-AI'

# Clone repo
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo đã tồn tại tại {REPO_DIR}, bỏ qua clone.")
    !cd {REPO_DIR} && git pull

# Thêm vào Python path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Cài dependencies
print("\nCài đặt dependencies...")
!pip install -r {REPO_DIR}/requirements.txt -q

print("\nHoàn tất cài đặt!")

Cloning into '/content/MoodNote-AI'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 49 (delta 7), reused 44 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 32.18 KiB | 8.04 MiB/s, done.
Resolving deltas: 100% (7/7), done.

Cài đặt dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 46.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 94.9 MB/s eta 0:00:00

Hoàn tất cài đặt!


## Cell 4 — Cấu hình paths cho Colab

In [4]:
import os

# --- Paths ---
REPO_DIR       = '/content/MoodNote-AI'
CONFIG_DIR     = f'{REPO_DIR}/configs'
RAW_DIR        = f'{REPO_DIR}/data/raw'
PROCESSED_DIR  = f'{REPO_DIR}/data/processed'
CHECKPOINT_DIR = f'{DRIVE_BASE}/checkpoints'
BEST_MODEL_DIR = f'{DRIVE_BASE}/best_model'

os.makedirs(RAW_DIR,       exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# --- Tắt W&B ---
os.environ['WANDB_MODE']    = 'disabled'
os.environ['WANDB_SILENT']  = 'true'

print("Paths đã cấu hình:")
print(f"  Config     : {CONFIG_DIR}")
print(f"  Data raw   : {RAW_DIR}")
print(f"  Processed  : {PROCESSED_DIR}")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print(f"  Best model : {BEST_MODEL_DIR}")
print("  W&B        : disabled")

Paths đã cấu hình:
  Config     : /content/MoodNote-AI/configs
  Data raw   : /content/MoodNote-AI/data/raw
  Processed  : /content/MoodNote-AI/data/processed
  Checkpoints: /content/drive/MyDrive/MoodNote-AI/checkpoints
  Best model : /content/drive/MyDrive/MoodNote-AI/best_model
  W&B        : disabled


## Cell 5 — Download & Preprocess dataset UIT-VSMEC

In [6]:
import os
os.chdir(REPO_DIR)

from src.data.download_dataset import download_uit_vsmec
from src.data.preprocess import preprocess_dataset

# Download UIT-VSMEC từ HuggingFace
print("=" * 50)
print("Bước 1: Download dataset")
print("=" * 50)
download_uit_vsmec(output_dir=RAW_DIR)

# Preprocess: word segmentation với pyvi
print("\n" + "=" * 50)
print("Bước 2: Preprocess (word segmentation)")
print("=" * 50)
preprocess_dataset(
    input_dir=RAW_DIR,
    output_dir=PROCESSED_DIR,
    config_path=f'{CONFIG_DIR}/model_config.yaml'
)

# Kiểm tra
import pandas as pd
for split in ['train', 'validation', 'test']:
    f = f'{PROCESSED_DIR}/{split}.csv'
    df = pd.read_csv(f)
    print(f"  {split:12s}: {len(df):,} mẫu — {f}")

Bước 1: Download dataset
Dataset loaded successfully!
Train samples: 5548
Validation samples: 686
Test samples: 693
Saved train split to /content/MoodNote-AI/data/raw/train.csv

Sample from train:
                                            Sentence  Emotion
0              cho mình xin bài nhạc tên là gì với ạ    Other
1  cho đáng đời con quỷ . về nhà lôi con nhà mày ...  Disgust

Saved validation split to /content/MoodNote-AI/data/raw/validation.csv

Sample from validation:
                                            Sentence    Emotion
0  tính tao tao biết , chẳng có chuyện gì có thể ...      Other
1           lại là lào cai , tự hào quê mình quá :))  Enjoyment

Saved test split to /content/MoodNote-AI/data/raw/test.csv

Sample from test:
                                    Sentence   Emotion
0           người ta có bạn bè nhìn vui thật   Sadness
1  cho nghỉ viêc mói đúng sao goi là kỷ luật  Surprise


Emotion distribution in training set:
Error downloading dataset: 'labels'
Please m

KeyError: 'labels'

## Cell 6 — Train model

Quá trình train ~5 epoch, khoảng **30-60 phút** trên T4 GPU.

In [ ]:
import os, torch
os.chdir(REPO_DIR)

from src.data.dataset import EmotionDataset
from src.models.phobert_classifier import PhoBERTEmotionClassifier
from src.models.model_utils import save_model, get_device, print_model_summary
from src.training.trainer import train_model
from src.utils.config import load_all_configs
from src.utils.logger import setup_logger
from src.utils.metrics import compute_metrics, print_metrics, plot_confusion_matrix
import numpy as np
from pathlib import Path

logger = setup_logger(name='colab_training')

# Load configs
configs       = load_all_configs(CONFIG_DIR)
model_config  = configs['model']
training_config = configs['training']

logger.info(f"Model     : {model_config['model']['name']}")
logger.info(f"Epochs    : {training_config['training']['num_epochs']}")
logger.info(f"Batch size: {training_config['training']['batch_size']}")
logger.info(f"LR        : {training_config['training']['learning_rate']}")

device = get_device()

# Datasets
model_name = model_config['model']['name']
max_len    = model_config['model']['max_seq_length']

train_dataset = EmotionDataset(f'{PROCESSED_DIR}/train.csv',      tokenizer_name=model_name, max_length=max_len)
val_dataset   = EmotionDataset(f'{PROCESSED_DIR}/validation.csv', tokenizer_name=model_name, max_length=max_len)
test_dataset  = EmotionDataset(f'{PROCESSED_DIR}/test.csv',       tokenizer_name=model_name, max_length=max_len)

logger.info(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# Model
model = PhoBERTEmotionClassifier(
    model_name=model_name,
    num_labels=model_config['model']['num_labels'],
    dropout=model_config['model']['dropout']
)
model.to(device)
print_model_summary(model)

# Train
trainer = train_model(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    training_config=training_config,
    output_dir=CHECKPOINT_DIR,
    use_wandb=False
)

# Evaluate on test set
logger.info("Evaluating on test set...")
predictions   = trainer.predict(test_dataset)
test_preds    = predictions.predictions
test_labels   = predictions.label_ids

detailed = compute_metrics(test_preds, test_labels)
print_metrics(detailed, model_config['emotion_labels'])

# Confusion matrix
cm_path = Path(CHECKPOINT_DIR) / 'confusion_matrix.png'
plot_confusion_matrix(test_preds, test_labels,
                      emotion_labels=model_config['emotion_labels'],
                      save_path=cm_path)

# Save best model vào Drive
save_model(
    model=trainer.model,
    tokenizer=train_dataset.tokenizer,
    save_dir=BEST_MODEL_DIR,
    config={
        'model_config': model_config,
        'training_config': training_config,
        'test_results': {
            'accuracy':    detailed['accuracy'],
            'f1_macro':    detailed['f1_macro'],
            'f1_weighted': detailed['f1_weighted']
        }
    }
)

print("\n" + "=" * 50)
print("TRAINING HOÀN TẤT")
print("=" * 50)
print(f"Accuracy   : {detailed['accuracy']:.4f}")
print(f"F1-Macro   : {detailed['f1_macro']:.4f}")
print(f"F1-Weighted: {detailed['f1_weighted']:.4f}")
print(f"Model đã lưu tại: {BEST_MODEL_DIR}")

## Cell 7 — Kiểm tra model & test predict

In [ ]:
import os
os.chdir(REPO_DIR)

from src.models.model_utils import load_model
from src.inference.predictor import EmotionPredictor

# Kiểm tra file đã lưu trong Drive
print("Files trong best_model:")
for f in sorted(os.listdir(BEST_MODEL_DIR)):
    size = os.path.getsize(f'{BEST_MODEL_DIR}/{f}') / 1024**2
    print(f"  {f:<30} {size:.1f} MB")

# Test predict
print("\nTest predict:")
predictor = EmotionPredictor(model_dir=BEST_MODEL_DIR)

test_sentences = [
    "Hôm nay tôi rất vui vì được nghỉ học!",
    "Tôi buồn quá, không biết phải làm sao.",
    "Thật tức giận khi bị đối xử bất công.",
    "Trời ơi, tin này làm tôi bất ngờ quá!",
]

for sentence in test_sentences:
    result = predictor.predict(sentence)
    print(f"  Text      : {sentence}")
    print(f"  Emotion   : {result['emotion']}")
    print(f"  Confidence: {result['confidence']:.2%}")
    print()